In [281]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, make_scorer
from sklearn.ensemble import IsolationForest

import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization
from keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

### 1. Загрузка и предобработка данных

In [90]:
df = pd.read_csv('data/bank_data_train.csv', index_col=0)

x = df.drop(columns=['TARGET'])
y = df['TARGET']

In [91]:
x.shape

(355190, 114)

In [92]:
x.info()

<class 'pandas.core.frame.DataFrame'>
Index: 355190 entries, 146841 to 590828
Columns: 114 entries, CR_PROD_CNT_IL to LDEAL_ACT_DAYS_PCT_CURR
dtypes: float64(94), int64(7), object(13)
memory usage: 311.6+ MB


In [93]:
# проверка и удаление пропусков
x.isna().sum() / len(x)

CR_PROD_CNT_IL              0.000000
AMOUNT_RUB_CLO_PRC          0.107894
PRC_ACCEPTS_A_EMAIL_LINK    0.563155
APP_REGISTR_RGN_CODE        0.829528
PRC_ACCEPTS_A_POS           0.563155
                              ...   
LDEAL_DELINQ_PER_MAXYWZ     0.730530
TURNOVER_DYNAMIC_CC_3M      0.000000
LDEAL_ACT_DAYS_PCT_TR       0.736907
LDEAL_ACT_DAYS_PCT_TR4      0.736907
LDEAL_ACT_DAYS_PCT_CURR     0.736907
Length: 114, dtype: float64

In [94]:
missing = x.isna().mean().sort_values(ascending=False)
drop_columns = missing[missing > 0.2].index
x = x.drop(drop_columns, axis=1)
print("Dropped columns:", drop_columns.tolist())

Dropped columns: ['CLNT_SALARY_VALUE', 'LDEAL_YQZ_COM', 'LDEAL_YQZ_CHRG', 'AVG_PCT_MONTH_TO_PCLOSE', 'MAX_PCLOSE_DATE', 'LDEAL_AMT_MONTH', 'AVG_PCT_DEBT_TO_DEAL_AMT', 'LDEAL_YQZ_PC', 'LDEAL_TENOR_MAX', 'DEAL_YQZ_IR_MIN', 'LDEAL_TENOR_MIN', 'LDEAL_DELINQ_PER_MAXYQZ', 'DEAL_YQZ_IR_MAX', 'LDEAL_USED_AMT_AVG_YQZ', 'MED_DEBT_PRC_YQZ', 'CLNT_JOB_POSITION_TYPE', 'APP_CAR', 'APP_DRIVING_LICENSE', 'APP_TRAVEL_PASS', 'APP_KIND_OF_PROP_HABITATION', 'APP_POSITION_TYPE', 'APP_REGISTR_RGN_CODE', 'CNT_TRAN_CLO_TENDENCY1M', 'SUM_TRAN_CLO_TENDENCY1M', 'APP_COMP_TYPE', 'APP_EMP_TYPE', 'APP_EDUCATION', 'APP_MARITAL_STATUS', 'CNT_TRAN_MED_TENDENCY1M', 'SUM_TRAN_MED_TENDENCY1M', 'CLNT_TRUST_RELATION', 'DEAL_GRACE_DAYS_ACC_AVG', 'DEAL_GRACE_DAYS_ACC_MAX', 'DEAL_GRACE_DAYS_ACC_S1X1', 'SUM_TRAN_AUT_TENDENCY1M', 'CNT_TRAN_AUT_TENDENCY1M', 'LDEAL_ACT_DAYS_ACC_PCT_AVG', 'LDEAL_ACT_DAYS_PCT_TR3', 'LDEAL_ACT_DAYS_PCT_TR', 'LDEAL_ACT_DAYS_PCT_TR4', 'LDEAL_ACT_DAYS_PCT_CURR', 'LDEAL_DELINQ_PER_MAXYWZ', 'LDEAL_USED_A

In [95]:
x.shape

(355190, 42)

In [100]:
x.info()

<class 'pandas.core.frame.DataFrame'>
Index: 355190 entries, 146841 to 590828
Data columns (total 42 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   CR_PROD_CNT_IL            355190 non-null  int64  
 1   AMOUNT_RUB_CLO_PRC        316867 non-null  float64
 2   TURNOVER_DYNAMIC_IL_1M    355190 non-null  float64
 3   AMOUNT_RUB_SUP_PRC        316867 non-null  float64
 4   REST_DYNAMIC_FDEP_1M      355190 non-null  float64
 5   REST_DYNAMIC_SAVE_3M      355190 non-null  float64
 6   CR_PROD_CNT_VCU           355190 non-null  int64  
 7   REST_AVG_CUR              355190 non-null  float64
 8   AMOUNT_RUB_NAS_PRC        316867 non-null  float64
 9   TRANS_COUNT_SUP_PRC       316867 non-null  float64
 10  TRANS_COUNT_NAS_PRC       316867 non-null  float64
 11  CR_PROD_CNT_TOVR          355190 non-null  int64  
 12  CR_PROD_CNT_PIL           355190 non-null  int64  
 13  TURNOVER_CC               355190 non-null  f

In [104]:
numeric_columns = x.select_dtypes(include='number').columns
category_columns = x.select_dtypes(include='object').columns
print(f'Numeric_columns: {len(numeric_columns)}, Category_columns: {len(category_columns)}')

Numeric_columns: 41, Category_columns: 1


In [119]:
X = x.copy()

imputer = SimpleImputer(missing_values=np.nan, strategy='median')
X[numeric_columns] = imputer.fit_transform(x[numeric_columns])

scaler = StandardScaler()
X[numeric_columns] = scaler.fit_transform(X[numeric_columns])

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder_values = encoder.fit_transform(x[category_columns])

encoder_df = pd.DataFrame(encoder_values, columns=encoder.get_feature_names_out(), index=X.index)

X = X.drop(columns=category_columns)
X = pd.concat([X, encoder_df], axis=1)

print(X.shape)

(355190, 53)


In [121]:
# 20% split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

In [123]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((284152, 53), (71038, 53), (284152,), (71038,))

In [125]:
# удаление выбросов и аномалий

In [127]:
iso_forest = IsolationForest(contamination=0.005)  # Предполагаем, что аномалий около 0.5%
iso_forest.fit(X_train)

# Получение аномальных оценок
anomalies_scores = iso_forest.decision_function(X_train)
anomalies_mask = anomalies_scores < np.percentile(anomalies_scores, 99)  # Порог выше 99%

# Фильтрация датасета
X_train = X_train[anomalies_mask]
y_train = y_train[anomalies_mask]

print(X_train.shape, y_train.shape)

(281310, 53) (281310,)


In [129]:
# Смотрим бисбаланс
y.value_counts(normalize=True)

TARGET
0    0.918565
1    0.081435
Name: proportion, dtype: float64

#### Pivot table result

In [134]:
result = pd.DataFrame(columns=['Model', 'Params', 'Accuracy', 'ROC AUC'])

def write_results(model_name, model, y_true, y_pred, y_pred_proba):
    try:
        params = model.get_params()
    except Exception:
        params = 'None'

    accuracy = accuracy_score(y_true, y_pred)
    roc_auc = roc_auc_score(y_true, y_pred_proba)

    print(f'Accuracy {accuracy}')
    print(f'ROC AUC {roc_auc}')

    global result

    if model_name in result['Model'].values:
        result.loc[result['Model'] == model_name, ['Params', 'Accuracy', 'ROC AUC']] = [params, accuracy, roc_auc]
    else:
        result = pd.concat([result, pd.DataFrame([{
            'Model': model_name,
            'Params': params,
            'Accuracy': accuracy,
            'ROC AUC': roc_auc
        }])], ignore_index=True)

In [136]:
result

,Model,Params,Accuracy,ROC AUC


### BaseLine

In [138]:
# Наивный классификатор
classifier = DummyClassifier(strategy='most_frequent')
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
y_pred_proba = classifier.predict_proba(X_test)[:, 1]
write_results('DummyClassifier', classifier, y_test, y_pred, y_pred_proba)

Accuracy 0.918564711844365
ROC AUC 0.5


### Random Forest

In [145]:
# RandomForest
params = {
    'class_weight': ['balanced', None],
    'n_estimators': [100, 200],
    'max_depth': [None, 10]
}

auc_scorer = make_scorer(roc_auc_score, needs_proba=True)

classifier = RandomForestClassifier(random_state=21, n_jobs=-1)
grid = RandomizedSearchCV(estimator=classifier, 
                          param_distributions=params, 
                          n_jobs=-1, 
                          verbose=2, 
                          random_state=21,
                          n_iter=8, 
                          cv=3,
                          scoring=auc_scorer)

grid.fit(X_train, y_train)
y_pred = grid.predict(X_test)
y_pred_proba = grid.predict_proba(X_test)[:, 1]
write_results('RandomForestClassifier', grid.best_estimator_, y_test, y_pred, y_pred_proba)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Accuracy 0.9187195585461302
ROC AUC 0.8226922227228555


In [146]:
result

,Model,Params,Accuracy,ROC AUC
0,DummyClassifier,"{'constant': None, 'random_state': None, 'stra...",0.918565,0.500000
1,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.918720,0.822692


### MLP

In [150]:
# MLPClassifier
params = {
    'solver': ['adam'],
    'alpha': [1e-4, 10e-4, 20e-4],
    'hidden_layer_sizes': [(32, 32), (64, 64), (64, 32), (128, 64)],
    'learning_rate_init': [0.001, 0.005]
}

auc_scorer = make_scorer(roc_auc_score, needs_proba=True)

mlp = MLPClassifier(random_state=21, activation='relu', max_iter=100, early_stopping=True, n_iter_no_change=15, verbose=True)

grid = RandomizedSearchCV(
    estimator=mlp,
    param_distributions=params,
    n_iter=8,
    cv=3,
    n_jobs=-1,
    verbose=2,
    random_state=21,
    scoring=auc_scorer
)

grid.fit(X_train, y_train)
y_pred = grid.predict(X_test)
y_pred_proba = grid.predict_proba(X_test)[:, 1]
write_results('MLPClassifier', grid.best_estimator_, y_test, y_pred, y_pred_proba)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Iteration 1, loss = 0.26307959
Validation score: 0.918240
Iteration 2, loss = 0.24444849
Validation score: 0.918275
Iteration 3, loss = 0.23876761
Validation score: 0.918275
Iteration 4, loss = 0.23586713
Validation score: 0.918204
Iteration 5, loss = 0.23359776
Validation score: 0.918346
Iteration 6, loss = 0.23237620
Validation score: 0.918275
Iteration 7, loss = 0.23113867
Validation score: 0.918311
Iteration 8, loss = 0.23001810
Validation score: 0.918737
Iteration 9, loss = 0.22964722
Validation score: 0.918560
Iteration 10, loss = 0.22882484
Validation score: 0.918737
Iteration 11, loss = 0.22839465
Validation score: 0.918489
Iteration 12, loss = 0.22774901
Validation score: 0.918595
Iteration 13, loss = 0.22752348
Validation score: 0.918489
Iteration 14, loss = 0.22740529
Validation score: 0.918702
Iteration 15, loss = 0.22685729
Validation score: 0.918560
Iteration 16, loss = 0.22633304
Validation score: 0.918204
Itera

In [158]:
result

,Model,Params,Accuracy,ROC AUC
0,DummyClassifier,"{'constant': None, 'random_state': None, 'stra...",0.918565,0.500000
1,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.918720,0.822692
2,MLPClassifier,"{'activation': 'relu', 'alpha': 0.0001, 'batch...",0.918917,0.811870


### Keras

In [169]:
# Keras
input_dim = X_train.shape[1]

params = {
    'Dropout': [0.2, 0.3],
    'hidden_layer_sizes': [(64, 64), (64, 32), (128, 64)],
    'bs': [512]
}

model = None
roc_auc = 0
y_pred_k, y_pred_proba_k = None, None

for dropout in params['Dropout']:
    for hidden in params['hidden_layer_sizes']:
        for bs in params['bs']:
            keras = Sequential([
                Dense(hidden[0], activation='relu', input_dim=input_dim),
                Dropout(dropout),
                Dense(hidden[1], activation='relu'),
                Dropout(dropout),
                Dense(1, activation='sigmoid')
            ])
            
            keras.compile(
                optimizer=Adam(learning_rate=0.001),
                loss='binary_crossentropy',
                metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
            )
            
            keras.fit(X_train, y_train, epochs=100, batch_size=bs, verbose=0)
            y_pred_proba = keras.predict(X_test)
            y_pred = (y_pred_proba > 0.5).astype(int)
            roc_auc_cur = roc_auc_score(y_test, y_pred_proba)
            if roc_auc_cur > roc_auc:
                model = keras
                roc_auc = roc_auc_cur
                y_pred_k, y_pred_proba_k = y_pred, y_pred_proba
            print(f'params: dropout  {dropout}, hidden {hidden}, batch_size {bs}')
            print(f'AUC: {roc_auc_cur}\n')
            
write_results('Keras', model, y_test, y_pred_k, y_pred_proba_k)

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 547us/step
params: dropout  0.2, hidden (64, 64), batch_size 512
AUC: 0.8260404694864896

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 563us/step
params: dropout  0.2, hidden (64, 32), batch_size 512
AUC: 0.8283744392761208

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 564us/step
params: dropout  0.2, hidden (128, 64), batch_size 512
AUC: 0.8260780335342837

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 595us/step
params: dropout  0.3, hidden (64, 64), batch_size 512
AUC: 0.8292848270214672

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 555us/step
params: dropout  0.3, hidden (64, 32), batch_size 512
AUC: 0.8257059441569105

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 525us/step
params: dropout  0.3, hidden (128, 64), batch_size 512
AUC: 0.8281442892825863

Accuracy 0.9191700216785382
ROC AUC 0.8292848270214672


In [171]:
result

,Model,Params,Accuracy,ROC AUC
0,DummyClassifier,"{'constant': None, 'random_state': None, 'stra...",0.918565,0.500000
1,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.918720,0.822692
2,MLPClassifier,"{'activation': 'relu', 'alpha': 0.0001, 'batch...",0.918917,0.811870
3,Keras,None,0.919170,0.829285


In [179]:
model.summary()

Model: "sequential_21"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_63 (Dense)                     │ (None, 64)                  │           3,456 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_42 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_64 (Dense)                     │ (None, 64)                  │           4,160 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_43 (Dropout)                 │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_65 (Dense)                     │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 23,045 (90.02 KB)

 Trainable params: 7,681 (30.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 15,364 (60.02 KB)

In [181]:
input_dim = X_train.shape[1]

keras = Sequential([
    Dense(64, activation='relu', input_dim=input_dim),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

keras.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

keras.fit(X_train, y_train, epochs=100, batch_size=512, verbose=1)
y_pred_proba = keras.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)
write_results('Keras', keras, y_test, y_pred, y_pred_proba)

Epoch 1/100
550/550 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9060 - auc: 0.5670 - loss: 0.3258
Epoch 2/100
550/550 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9176 - auc: 0.6974 - loss: 0.2673
Epoch 3/100
550/550 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9180 - auc: 0.7265 - loss: 0.2597
Epoch 4/100
550/550 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9181 - auc: 0.7435 - loss: 0.2545
Epoch 5/100
550/550 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9183 - auc: 0.7604 - loss: 0.2496
Epoch 6/100
550/550 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9190 - auc: 0.7691 - loss: 0.2454
Epoch 7/100
550/550 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9183 - auc: 0.7764 - loss: 0.2443
Epoch 8/100
550/550 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9184 - auc: 0.7837 - loss: 0.2418
Epoch 9/100
550/550 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9180 - auc: 0.7865 - loss: 0.2423
Epoch 10/100
550/550 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9173 - auc: 0.7930 - lo

In [187]:
result.loc[result['Model'] == 'Keras', 'Params'] = [
    {'layers': [(X_train.shape[1], 64), (64, 64), (64, 1)], 'activation': 'relu', 'dropout': 0.3, 'optimizer': 'adam',
     'learning_rate': 0.001, 'loss': 'binary_crossentropy', 'batch_size': 512, 'epochs': 100}]

In [189]:
result

,Model,Params,Accuracy,ROC AUC
0,DummyClassifier,"{'constant': None, 'random_state': None, 'stra...",0.918565,0.500000
1,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.918720,0.822692
2,MLPClassifier,"{'activation': 'relu', 'alpha': 0.0001, 'batch...",0.918917,0.811870
3,Keras,"{'layers': [(53, 64), (64, 64), (64, 1)], 'act...",0.919212,0.827079


### TensorFlow

In [197]:
class TFMLP:
    def __init__(self, input_dim):
        self.W1 = tf.Variable(tf.random.truncated_normal([input_dim, 64], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([64]))

        self.W2 = tf.Variable(tf.random.truncated_normal([64, 64], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([64]))

        self.W3 = tf.Variable(tf.random.truncated_normal([64, 1], stddev=0.1))
        self.b3 = tf.Variable(tf.zeros([1]))

        self.optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

    def forward(self, X, training=True):

        x = tf.matmul(X, self.W1) + self.b1
        x = tf.nn.relu(x)
        if training:
            x = tf.nn.dropout(x, rate=0.3)
        x = tf.matmul(x, self.W2) + self.b2
        x = tf.nn.relu(x)
        if training:
            x = tf.nn.dropout(x, rate=0.3)
        x = tf.matmul(x, self.W3) + self.b3
        y_hat = tf.sigmoid(x)
        return y_hat

    def compute_loss(self, y_true, y_pred):
        return tf.reduce_mean(tf.keras.losses.binary_crossentropy(y_true, y_pred))

    def train_step(self, X_batch, y_batch):
        with tf.GradientTape() as tape:
            y_pred = self.forward(X_batch, training=True)
            loss = self.compute_loss(y_batch, y_pred)
        grads = tape.gradient(loss, [self.W1, self.b1, self.W2, self.b2, self.W3, self.b3])
        self.optimizer.apply_gradients(zip(grads, [self.W1, self.b1, self.W2, self.b2, self.W3, self.b3]))
        return loss

    def predict_proba(self, X):
        return self.forward(X, training=False).numpy()

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) > threshold).astype(int)

    def get_params(self):
        return {'input_dim': X_train.shape[1], 'hidden_layers': [64, 64], 'activation': 'relu', 'dropout': 0.3,
     'optimizer': 'adam', 'learning_rate': 0.001, 'loss': 'binary_crossentropy', 'batch_size': 516, 'epochs': 100,
     'early_stopping': True, 'patience': 10}

In [199]:
X_train_tf = tf.convert_to_tensor(X_train.values, dtype=tf.float32)
y_train_tf = tf.convert_to_tensor(y_train.values.reshape(-1, 1), dtype=tf.float32)

X_test_tf = tf.convert_to_tensor(X_test.values, dtype=tf.float32)
y_test_tf = y_test.values

batch_size = 516
epochs = 100

dataset_tf = tf.data.Dataset.from_tensor_slices((X_train_tf, y_train_tf))
dataset_tf = dataset_tf.shuffle(buffer_size=X_train.shape[0]).batch(batch_size)

tfmpl = TFMLP(input_dim=X_train.shape[1])

patience = 10
best_auc = 0
epochs_without_improve = 0
best_weights = None

for epoch in range(epochs):
    for X_batch, y_batch in dataset_tf:
        tfmpl.train_step(X_batch, y_batch)

    y_pred_proba = tfmpl.predict_proba(X_test_tf)
    auc = roc_auc_score(y_test, y_pred_proba)
    if epoch % 5 == 0:
        print(f"Epoch {epoch + 1}: ROC AUC = {auc:.4f}")
    if auc > best_auc:
        best_auc = auc
        epochs_without_improve = 0
        best_weights = [
            tf.identity(tfmpl.W1),
            tf.identity(tfmpl.b1),
            tf.identity(tfmpl.W2),
            tf.identity(tfmpl.b2),
            tf.identity(tfmpl.W3),
            tf.identity(tfmpl.b3),
        ]
    else:
        epochs_without_improve += 1

    if epochs_without_improve >= patience:
        print("Early stopping")
        break

    if best_weights is not None:
        tfmpl.W1.assign(best_weights[0])
        tfmpl.b1.assign(best_weights[1])
        tfmpl.W2.assign(best_weights[2])
        tfmpl.b2.assign(best_weights[3])
        tfmpl.W3.assign(best_weights[4])
        tfmpl.b3.assign(best_weights[5])

y_pred = tfmpl.predict(X_test_tf)
y_pred_proba = tfmpl.predict_proba(X_test_tf)
write_results('TFMLP', tfmpl, y_test_tf, y_pred, y_pred_proba)

Epoch 1: ROC AUC = 0.7262
Epoch 6: ROC AUC = 0.7902
Epoch 11: ROC AUC = 0.8091
Epoch 16: ROC AUC = 0.8140
Epoch 21: ROC AUC = 0.8169
Epoch 26: ROC AUC = 0.8147
Epoch 31: ROC AUC = 0.8175
Early stopping
Accuracy 0.9189166361665587
ROC AUC 0.8163110221565496


In [200]:
result

,Model,Params,Accuracy,ROC AUC
0,DummyClassifier,"{'constant': None, 'random_state': None, 'stra...",0.918565,0.500000
1,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.918720,0.822692
2,MLPClassifier,"{'activation': 'relu', 'alpha': 0.0001, 'batch...",0.918917,0.811870
3,Keras,"{'layers': [(53, 64), (64, 64), (64, 1)], 'act...",0.919212,0.827079
4,TFMLP,"{'input_dim': 53, 'hidden_layers': [64, 64], '...",0.918917,0.816311


### NumPy

In [210]:
class NumpyMLP:
    def __init__(self, input_dim, lr=0.01):
        self.lr = lr

        self.W1 = np.random.randn(input_dim, 64) * 0.1
        self.b1 = np.zeros((1, 64))

        self.W2 = np.random.randn(64, 64) * 0.1
        self.b2 = np.zeros((1, 64))

        self.W3 = np.random.randn(64, 1) * 0.1
        self.b3 = np.zeros((1, 1))

    def relu(self, X):
        return np.maximum(0, X)

    def sigmoid(self, X):
        return 1 / (1 + np.exp(-X))

    def relu_deriv(self, X):
        return (X > 0).astype(float)

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = self.relu(self.z1)

        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self.relu(self.z2)

        self.z3 = self.a2 @ self.W3 + self.b3
        self.y_hat = self.sigmoid(self.z3)

        return self.y_hat

    def compute_loss(self, y_true, y_pred):
        eps = 1e-8
        loss = -(y_true * np.log(y_pred + eps) +
                 (1 - y_true) * np.log(1 - y_pred + eps))
        return np.mean(loss)

    def backward(self, X, y):
        m = X.shape[0]

        dz3 = self.y_hat - y
        dW3 = (self.a2.T @ dz3) / m
        db3 = np.mean(dz3, axis=0, keepdims=True)

        da2 = dz3 @ self.W3.T
        dz2 = da2 * self.relu_deriv(self.z2)

        dW2 = (self.a1.T @ dz2) / m
        db2 = np.mean(dz2, axis=0, keepdims=True)

        da1 = dz2 @ self.W2.T
        dz1 = da1 * self.relu_deriv(self.z1)

        dW1 = (X.T @ dz1) / m
        db1 = np.mean(dz1, axis=0, keepdims=True)

        self.W3 -= self.lr * dW3
        self.b3 -= self.lr * db3

        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2

        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train_step(self, X, y):
        y_pred = self.forward(X)
        loss = self.compute_loss(y, y_pred)
        self.backward(X, y)

        return loss

    def predict_proba(self, X):
        return self.forward(X)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) > threshold).astype(int)

    def get_params(self):
        return {'input_dim': X_train.shape[1], 'hidden_layers': [64, 64], 'activation': 'relu', 'learning_rate': 0.01,
     'batch_size': 516, 'epochs': 100, 'early_stopping': True, 'patience': 10}

In [212]:
numpymlp = NumpyMLP(input_dim=X_train.shape[1], lr=0.01)

patience = 10
best_auc = 0
epochs_without_improve = 0
best_weights = None

batch_size = 516
epochs = 100

X_train_np = X_train.to_numpy(dtype=float)
y_train_np = y_train.to_numpy(dtype=float).reshape(-1, 1)
X_test_np = X_test.to_numpy(dtype=float)
y_test_np = y_test.to_numpy(dtype=float).reshape(-1, 1)

for epoch in range(epochs):

    indices = np.random.permutation(len(X_train))
    X_train_sh = X_train_np[indices]
    y_train_sh = y_train_np[indices].reshape(-1, 1)

    for i in range(0, len(X_train_sh), batch_size):
        X_batch = X_train_sh[i:i + batch_size]
        y_batch = y_train_sh[i:i + batch_size]

        loss = numpymlp.train_step(X_batch, y_batch)

    y_pred = numpymlp.predict_proba(X_test_np)
    auc = roc_auc_score(y_test, y_pred)

    if epoch % 5 == 0:
        print(f"Epoch {epoch + 1}: AUC = {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        epochs_without_improve = 0
        best_weights = [
            numpymlp.W1.copy(),
            numpymlp.b1.copy(),
            numpymlp.W2.copy(),
            numpymlp.b2.copy(),
            numpymlp.W3.copy(),
            numpymlp.b3.copy(),
        ]
    else:
        epochs_without_improve += 1

    if epochs_without_improve >= patience:
        print("Early stopping")
        break

    if best_weights is not None:
        numpymlp.W1 = best_weights[0]
        numpymlp.b1 = best_weights[1]

        numpymlp.W2 = best_weights[2]
        numpymlp.b2 = best_weights[3]

        numpymlp.W3 = best_weights[4]
        numpymlp.b3 = best_weights[5]

y_pred = numpymlp.predict(X_test_tf)
y_pred_proba = numpymlp.predict_proba(X_test_tf)
write_results('NumpyMLP', numpymlp, y_test_np, y_pred, y_pred_proba)

Epoch 1: AUC = 0.5212
Epoch 6: AUC = 0.6656
Epoch 11: AUC = 0.6968
Epoch 16: AUC = 0.7144
Epoch 21: AUC = 0.7269
Epoch 26: AUC = 0.7363
Epoch 31: AUC = 0.7439
Epoch 36: AUC = 0.7496
Epoch 41: AUC = 0.7543
Epoch 46: AUC = 0.7585
Epoch 51: AUC = 0.7616
Epoch 56: AUC = 0.7646
Epoch 61: AUC = 0.7672
Epoch 66: AUC = 0.7694
Epoch 71: AUC = 0.7715
Epoch 76: AUC = 0.7733
Epoch 81: AUC = 0.7750
Epoch 86: AUC = 0.7763
Epoch 91: AUC = 0.7780
Epoch 96: AUC = 0.7789
Accuracy 0.9188180973563445
ROC AUC 0.7806223925620219


In [213]:
result

,Model,Params,Accuracy,ROC AUC
0,DummyClassifier,"{'constant': None, 'random_state': None, 'stra...",0.918565,0.500000
1,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w...",0.918720,0.822692
2,MLPClassifier,"{'activation': 'relu', 'alpha': 0.0001, 'batch...",0.918917,0.811870
3,Keras,"{'layers': [(53, 64), (64, 64), (64, 1)], 'act...",0.919212,0.827079
4,TFMLP,"{'input_dim': 53, 'hidden_layers': [64, 64], '...",0.918917,0.816311
5,NumpyMLP,"{'input_dim': 53, 'hidden_layers': [64, 64], '...",0.918818,0.780622


In [216]:
result.to_csv('data/result.csv')

### Tuning best model

In [283]:
input_dim = X_train.shape[1]

params = {
    'Dropout': [0.2, 0.3],
    'hidden_layer_sizes': [(128, 64), (32, 32, 16), (32, 16, 16)],
    'bs': [1024, 2024],
    'lr': [0.001, 0.0001]
}

model = None
roc_auc = 0
y_pred_k, y_pred_proba_k = None, None

n = 1
for v in params.values():
    n *= len(v)
print(f'N params = {n}')

es = EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=10,
    restore_best_weights=True,
    verbose=0
)

best_params = {}
for dropout in params['Dropout']:
    for hidden in params['hidden_layer_sizes']:
        for bs in params['bs']:
            for lr in params['lr']:
                if len(hidden) == 3:
                    keras = Sequential([
                        Dense(hidden[0], activation='relu', input_dim=input_dim),
                        BatchNormalization(),
                        Dropout(dropout),
                        Dense(hidden[1], activation='relu'),
                        BatchNormalization(),
                        Dropout(dropout),
                        Dense(hidden[2], activation='relu'),
                        BatchNormalization(),
                        Dropout(dropout),
                        Dense(1, activation='sigmoid')
                    ])
                elif len(hidden) == 2:
                    keras = Sequential([
                        Dense(hidden[0], activation='relu', input_dim=input_dim),
                        BatchNormalization(),
                        Dropout(dropout),
                        Dense(hidden[1], activation='relu'),
                        BatchNormalization(),
                        Dropout(dropout),
                        Dense(1, activation='sigmoid')
                    ])
                
                keras.compile(
                    optimizer=Adam(learning_rate=lr),
                    loss='binary_crossentropy',
                    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
                )
                
                keras.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=200, batch_size=bs, verbose=0, callbacks=[es])
                y_pred_proba = keras.predict(X_test)
                y_pred = (y_pred_proba > 0.5).astype(int)
                roc_auc_cur = roc_auc_score(y_test, y_pred_proba)
                if roc_auc_cur > roc_auc:
                    model = keras
                    roc_auc = roc_auc_cur
                    y_pred_k, y_pred_proba_k = y_pred, y_pred_proba
                    best_params = {
                        'Dropout': dropout,
                        'hidden_layer_sizes': hidden,
                        'bs': bs,
                        'lr': lr,
                        'epochs': 200,
                        'activation': 'relu',
                        'optimizer': 'adam'
                    }
                print(f'params: dropout  {dropout}, hidden {hidden}, batch_size {bs}')
                print(f'AUC: {roc_auc_cur}\n')


# write_results('Keras', model, y_test, y_pred_k, y_pred_proba_k)

N params = 24
2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 634us/step
params: dropout  0.2, hidden (128, 64), batch_size 1024
AUC: 0.832212125979273

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 635us/step
params: dropout  0.2, hidden (128, 64), batch_size 1024
AUC: 0.832742949949443

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 626us/step
params: dropout  0.2, hidden (128, 64), batch_size 2024
AUC: 0.8302780000471802

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 600us/step
params: dropout  0.2, hidden (128, 64), batch_size 2024
AUC: 0.8303230199491718

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 607us/step
params: dropout  0.2, hidden (32, 32, 16), batch_size 1024
AUC: 0.8315073205984589

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 604us/step
params: dropout  0.2, hidden (32, 32, 16), batch_size 1024
AUC: 0.8276023364996674

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 624us/step
params: dropout  0.2, hidden (32, 32, 16), batch_size 2024
AUC: 0.8307485758411172

2220/2220 ━━━━━━━━━━━━━━━━━━━━ 1s 617us/step
params: dropout  0.2, hidden (32, 32, 16), batch_siz

In [284]:
model.summary()

Model: "sequential_74"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_262 (Dense)                    │ (None, 128)                 │           6,912 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_34               │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_187 (Dropout)                │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_263 (Dense)                    │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_35               │ (None, 64)                  │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_188 (Dropout)                │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_264 (Dense)                    │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 47,237 (184.52 KB)

 Trainable params: 15,617 (61.00 KB)

 Non-trainable params: 384 (1.50 KB)

 Optimizer params: 31,236 (122.02 KB)

In [285]:
best_params

{'Dropout': 0.3,
 'hidden_layer_sizes': (128, 64),
 'bs': 1024,
 'lr': 0.0001,
 'epochs': 200,
 'activation': 'relu',
 'optimizer': 'adam'}

In [295]:
input_dim = X_train.shape[1]

es = EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=10,
    restore_best_weights=True,
    verbose=0
)

keras = Sequential([
    Dense(128, activation='relu', input_dim=input_dim),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

keras.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

keras.fit(X, y, epochs=200, batch_size=1024, verbose=1, callbacks=[es])

Epoch 1/200
347/347 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.5492 - auc: 0.5445 - loss: 0.8082
Epoch 2/200
347/347 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.7136 - auc: 0.5609 - loss: 0.6199
Epoch 3/200
347/347 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8277 - auc: 0.5753 - loss: 0.5060
Epoch 4/200
347/347 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8800 - auc: 0.5904 - loss: 0.4189
Epoch 5/200
347/347 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9009 - auc: 0.5994 - loss: 0.3595
Epoch 6/200
347/347 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9078 - auc: 0.6204 - loss: 0.3237
Epoch 7/200
347/347 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9123 - auc: 0.6379 - loss: 0.3036
Epoch 8/200
347/347 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9134 - auc: 0.6527 - loss: 0.2934
Epoch 9/200
347/347 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9146 - auc: 0.6667 - loss: 0.2873
Epoch 10/200
347/347 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9149 - auc: 0.6734 - lo

In [296]:
model.save('data/my_model.keras')

In [299]:
df_test = pd.read_csv('data/bank_data_test.csv', index_col=0)

all_columns = numeric_columns.to_list() + category_columns.to_list()
x = df_test.drop(columns=['TARGET'])[all_columns]
y = df_test['TARGET']

# Преобразование данных
numeric_columns = x.select_dtypes(include='number').columns
category_columns = x.select_dtypes(include='object').columns
print(f'Numeric_columns: {len(numeric_columns)}, Category_columns: {len(category_columns)}')

X = x.copy()

imputer = SimpleImputer(missing_values=np.nan, strategy='median')
X[numeric_columns] = imputer.fit_transform(x[numeric_columns])

scaler = StandardScaler()
X[numeric_columns] = scaler.fit_transform(X[numeric_columns])

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder_values = encoder.fit_transform(x[category_columns])

encoder_df = pd.DataFrame(encoder_values, columns=encoder.get_feature_names_out(), index=X.index)

X = X.drop(columns=category_columns)
X = pd.concat([X, encoder_df], axis=1)

X = X.reindex(columns=X_train.columns, fill_value=0)

print(X.shape)

Numeric_columns: 41, Category_columns: 1
(88798, 53)


In [322]:
y_pred_proba = keras.predict(X)
y_pred = (y_pred_proba > 0.5).astype(int)

2775/2775 ━━━━━━━━━━━━━━━━━━━━ 2s 631us/step


In [326]:
pred_df = pd.DataFrame(y_pred_proba, columns=['TARGET'], index=X.index)
pred_df.to_csv('data/predict.csv')